**Assignment #2: Text Features and Rating Prediction (100 points)**

- **Instructor:** Jaeung Sim (University of Connecticut)
- **Course:** OPIM 5671 Data Mining and Time Series Forecasting
- **Last updated:** October 27, 2026

Companion: `05_Text_Features_and_Rating_Prediction_2026-10-27.ipynb`.
Use Google Colab's default CPU runtime. Complete TODO cells and replace response
prompts with your answers. Provided helpers are available for use.
Submit this completed `.ipynb` with outputs and written explanations.
Release and due dates will be announced by the instructor.

**Objective:** Determine whether sentiment, normalized emotion features and topics
improve predictions of Disneyland review ratings. Make decisions on training and
validation data, then assess the chosen model once on the held-out test.

| Part | Points |
|---|---:|
| I. Data preparation and splitting | 20 |
| II. Sentiment | 10 |
| III. Emotions and length | 15 |
| IV. Topic modeling (required) | 20 |
| V. Prediction and evaluation | 25 |
| VI. Interpretation | 10 |
| Total | 100 |


## Setup and provided helpers
Run these cells as provided. Use the same runtime limits as Notebook 06:
3,000 reviews, 1,200 topic-fitting reviews, 150 words per topic document,
1,500 vocabulary terms, K in {2,3,4}, and five LDA iterations.
Keep SEED = 25. Do not increase the search or add transformer models.


In [ ]:
%pip -q install textblob==0.19.0 nrclex==4.1.0 kagglehub


In [ ]:
import re, time, random, importlib.metadata
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from textblob import TextBlob
from nrclex import NRCLex
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 25
MAX_REVIEWS = 3000
TOPIC_FIT_LIMIT = 1200
TOPIC_WORD_LIMIT = 150
TOPIC_COUNTS = (2, 3, 4)
EMOTIONS = ['anger','fear','anticipation','trust','surprise','sadness','joy','disgust']
EMOTION_COLS = [f'{e}_per100' for e in EMOTIONS]
for package in ['numpy','pandas','scikit-learn','textblob','nrclex']:
    print(package, importlib.metadata.version(package))


In [ ]:
# Automatic Kaggle download. If unavailable, upload DisneylandReviews.csv
# to Colab's Files pane and set LOCAL_CSV to its /content/... path.
LOCAL_CSV = None
if LOCAL_CSV:
    csv_path = Path(LOCAL_CSV)
else:
    import kagglehub
    data_path = Path(kagglehub.dataset_download('arushchillar/disneyland-reviews'))
    csv_files = sorted(data_path.rglob('*.csv'))
    if not csv_files:
        raise FileNotFoundError('Upload DisneylandReviews.csv and set LOCAL_CSV.')
    preferred = [p for p in csv_files if 'disneyland' in p.name.lower()]
    csv_path = (preferred or csv_files)[0]
raw = pd.read_csv(csv_path, encoding='latin-1')
required = {'Review_Text','Rating'}
assert required.issubset(raw.columns), f'Missing columns: {required - set(raw.columns)}'
print('Loaded:', csv_path.name, raw.shape)


In [ ]:
def word_tokens(text):
    # Lightweight lexical matching. No NLTK corpus downloads are needed.
    return re.findall(r"[a-z]+(?:'[a-z]+)?", str(text).lower().replace('’', "'"))

TOPIC_STOPS = set(ENGLISH_STOP_WORDS) | {'disney','disneyland','park'}
def clean_for_topics(text):
    words = word_tokens(text)[:TOPIC_WORD_LIMIT]
    return ' '.join(w for w in words if len(w)>2 and w not in TOPIC_STOPS)

# Topic cleaning is deterministic. Learn the vocabulary later from training only.
# Sentiment receives the original text, preserving negation and punctuation.


In [ ]:
def top_words(model, vectorizer, n=8):
    terms = vectorizer.get_feature_names_out()
    return {f'Topic_{i+1}':list(terms[row.argsort()[-n:][::-1]])
            for i,row in enumerate(model.components_)}

def umass_coherence(model, counts, top_n=8):
    # UMass uses document co-occurrence among each topic's top words.
    # Higher (less negative) is better; this is NOT gensim's c_v score.
    present = (counts > 0).astype(np.int32).tocsc()
    scores = []
    for weights in model.components_:
        ids = weights.argsort()[-top_n:][::-1]
        pairs = []
        for m in range(1,len(ids)):
            for l in range(m):
                together = present[:,ids[m]].multiply(present[:,ids[l]]).sum()
                earlier = present[:,ids[l]].sum()
                pairs.append(np.log((float(together)+1)/max(float(earlier),1)))
        scores.append(np.mean(pairs))
    return float(np.mean(scores))

def new_lda(k):
    return LatentDirichletAllocation(n_components=k, learning_method='batch',
        max_iter=5, max_doc_update_iter=20, evaluate_every=-1,
        n_jobs=1, random_state=SEED)

def metrics(y, prediction):
    return {'MAE':mean_absolute_error(y,prediction),
            'MSE':mean_squared_error(y,prediction), 'R2':r2_score(y,prediction)}


## Part I. Data preparation and splitting (20 points)
**I-1 (10):** Remove missing/blank text and invalid ratings. Remove duplicate
review texts, take a reproducible sample of at most 3,000 reviews, and reset the index.
Preserve original text. Name the result `df`.


In [ ]:
# TODO I-1: Create df. Follow the cleaning rules above.
raise NotImplementedError('Complete I-1')


**I-2 (10):** Reserve 20% for test. Split the remaining development data into
80% training and 20% validation. Use `random_state=SEED` and stratify by rating
for both splits. Name the partitions `dev`, `train`, `valid`, `test`.
Print sizes and confirm disjoint indices. Explain each partition's role.
Do not inspect test ratings or performance before Part V-3.


In [ ]:
# TODO I-2: Create dev, train, valid, test and check disjointness.
raise NotImplementedError('Complete I-2')


**Response I-2:** Explain how this split prevents using test results to choose the model.


## Part II. Sentiment (10 points)
Generate TextBlob polarity from the **original** review text. Define
`sentiment_features(frame)` returning a DataFrame with column `TextBlob_Sentiment`
and the same index as `frame`. Apply it to train and valid as `sent_train` and
`sent_valid`. Compare “good” and “not good” and explain why topic-cleaned text
should not automatically be used for sentiment.


Use the same function interface demonstrated in Section 3 of Notebook 06.


In [ ]:
# TODO II: Define sentiment_features(frame), sent_train, sent_valid.
raise NotImplementedError('Complete II')


**Response II:** Explain the negation example and one limitation of polarity.


## Part III. Emotion rates and review length (15 points)
Define `emotion_features(frame)` returning `Review_Length` and the eight
`EMOTION_COLS`, preserving the input index. Use `word_tokens` and NRCLex's
`load_token_list`. For emotion e compute **100 × count(e) / max(token count,1)**.
Include zero for an absent emotion. Do not use `affect_frequencies`, which uses
a different denominator. Load the lexicon once outside the row loop.
Combine sentiment and emotion/length features as `base_train` and `base_valid`.
Explain why rates and length are separate predictors.


Notebook 06 demonstrates this same function and joins it to sentiment features.


In [ ]:
# TODO III: Define emotion_features and build base_train, base_valid.
raise NotImplementedError('Complete III')


**Response III:** Explain normalization, the denominator, and one context limitation of lexical emotion matching.


## Part IV. Required topic modeling (20 points)
**IV-1 (5):** Sample at most `TOPIC_FIT_LIMIT` training rows as `topic_fit`.
Fit `CountVectorizer` only on their original text using `clean_for_topics`,
`max_features=1500`, `min_df=3`, `max_df=0.90`, `tokenizer=str.split`,
`token_pattern=None` and `lowercase=False`. This uses the cleaned tokens
unchanged, preserving contractions rather than creating new word fragments.
Create `counts_fit`, `counts_train`, `counts_valid`.
Use word counts, not TF–IDF. Do not fit on validation or test.


In [ ]:
# TODO IV-1: Create topic_fit, vectorizer and the three count matrices.
raise NotImplementedError('Complete IV-1')


**IV-2 (10):** Use provided `new_lda` and `umass_coherence` helpers to fit only
K = 2, 3, 4. Store models in `topic_models`, and scores in `topic_scores` with
columns `K`, `UMass`, `Seconds`. Plot K against coherence and display top words.
Choose `chosen_k` using coherence and interpretability. Justify the choice in
2–3 sentences. These are UMass scores, not c_v scores.


Topics may overlap or be hard to name. The lecture's theme labels are illustrative;
you do not need to reproduce them. Choose K using the available evidence rather
than forcing clear separation.


In [ ]:
# TODO IV-2: Fit the bounded topic search, plot scores and set chosen_k.
raise NotImplementedError('Complete IV-2')


**Response IV-2:** State K and explain its coherence and interpretability.
For at least two topics, give tentative labels if supported by their top words,
or explain why their word lists overlap too much for confident labels.


**IV-3 (5):** Use the selected model to transform training and validation count
matrices into `topic_train` and `topic_valid`. Name columns `Topic_1` through
`Topic_K` and preserve the review indices. Check each row sums to one.
Show two high-weight training reviews for each of two topics, and explain whether
they support your tentative interpretation or your conclusion that the
topics are ambiguous. Keep all weights for inspection; later use K−1 in regression.


Grading emphasizes correct extraction and evidence from words/reviews. A
well-supported explanation of ambiguity receives the same interpretation credit
as a well-supported label; no particular topic names or separation are required.


In [ ]:
# TODO IV-3: Create lda, topic_cols, topic_train, topic_valid and inspect examples.
raise NotImplementedError('Complete IV-3')


**Response IV-3:** Explain how the example reviews support or challenge your
tentative interpretations, including any evidence of overlapping or ambiguous topics.


## Part V. Prediction and evaluation (25 points)
**V-1 (5):** Build `X_train`, `X_valid` and `feature_sets` for the same three
regression models as Notebook 06. Include length with emotion rates and omit the
last topic weight from the full model. Confirm indices align and values are finite.
Explain the omitted topic weight.


In [ ]:
# TODO V-1: Assemble X_train, X_valid and feature_sets.
raise NotImplementedError('Complete V-1')


**Response V-1:** Why does including all K topic weights with an intercept create redundancy?


**V-2 (10):** Fit the training-mean baseline and all three regressions on training
rows. Produce one `comparison` table with training and validation MAE, MSE and R²
for every model. Select `selected_name` by lowest validation MAE, including the
baseline. For exact ties favor the earlier, simpler model. Freeze this choice and K.
Explain the validation winner and the training/validation gap. No test evaluation yet.


In [ ]:
# TODO V-2: Fit, compare and freeze selected_name.
raise NotImplementedError('Complete V-2')


**Response V-2:** Record your selected feature set and K before continuing. Explain whether the extra features helped on validation.


**V-3 (10): Final test.** Run the provided cell only after freezing your choices.
It keeps the training-fitted vocabulary/LDA fixed, refits the chosen regression
on development rows, and evaluates the selected model plus a development-mean
baseline once on test. Report MAE, MSE and R², interpret the change from validation,
and inspect the largest errors. Do not choose a new model after this cell.
Use raw predictions consistently, without rounding or clipping.


The final reporting cell has no saved test outputs. Record your choices in
Response V-2 before running it.


In [ ]:
# Adapter for the final reporting cell. Your functions must preserve row indices.
def base_features(frame):
    return sentiment_features(frame).join(emotion_features(frame))

# Run after freezing K and selected_name. Do not change them based on test results.
# Keep the training-fitted vocabulary and LDA fixed; refit regression on dev.
X_dev = pd.concat([X_train,X_valid]).sort_index()
y_dev = df.loc[X_dev.index,'Rating']
base_test = base_features(test)
counts_test = vectorizer.transform(test['Review_Text'])
topic_test = pd.DataFrame(lda.transform(counts_test),columns=topic_cols,index=test.index)
X_test = base_test.join(topic_test)
final_baseline = DummyRegressor(strategy='mean').fit(np.zeros((len(y_dev),1)),y_dev)
baseline_test_prediction = final_baseline.predict(np.zeros((len(test),1)))
if selected_name == 'Mean baseline':
    final_model = final_baseline
    final_prediction = baseline_test_prediction
else:
    selected_cols = feature_sets[selected_name]
    final_model = LinearRegression().fit(X_dev[selected_cols],y_dev)
    final_prediction = final_model.predict(X_test[selected_cols])
test_results = pd.DataFrame([
    {'Model':'Mean baseline',**metrics(test['Rating'],baseline_test_prediction)},
    {'Model':selected_name,**metrics(test['Rating'],final_prediction)}
]).drop_duplicates('Model')
display(test_results.round(4))
print('Predictions outside 1–5:', int(((final_prediction<1)|(final_prediction>5)).sum()))
# Evaluate raw regression predictions consistently. Do not round or clip them.
error_review = test[['Review_Text','Rating']].copy()
error_review['Prediction'] = final_prediction
error_review['Absolute_Error'] = abs(error_review['Rating']-final_prediction)
display(error_review.nlargest(5,'Absolute_Error'))


**Response V-3:** Interpret the selected model’s test results against the baseline. If the test result disappoints, explain it without switching models.


## Part VI. Recommendation and limitations (10 points)
Write 150–250 words addressing:
1. Your validation-selected model and whether it beat the baseline on test.
2. Whether added features justified their complexity; use reported numbers.
3. One difficult review and why the features may miss its meaning.
4. Two limitations, including the ordinal/bounded rating target or transfer to
   other review contexts. Distinguish prediction from causal explanation.

Sentiment+emotion+length adds two feature groups together. Do not attribute any
improvement solely to emotions without an additional controlled comparison.


**Response VI:** Replace this prompt with your recommendation.


## Submission checklist
- Complete every TODO and written response; include outputs from a fresh sequential run.
- Keep the specified seed, sample limits and topic candidates.
- Show coherence/top words, validation comparison, and final selected-model test results.
- Verify no text vectorizer or topic model was fitted on validation/test reviews.
- Report any changes or execution problems. No AIC/BIC or adjusted R² required.


## References
- [Disneyland reviews dataset](https://www.kaggle.com/datasets/arushchillar/disneyland-reviews)
- [TextBlob sentiment API](https://textblob.readthedocs.io/en/stable/api_reference.html)
- [NRCLex 4.1 documentation](https://pypi.org/project/NRCLex/4.1.0/)
- Mohammad, S. M. & Turney, P. D. (2013). Crowdsourcing a Word–Emotion Association Lexicon. Computational Intelligence, 29(3), 436–465. [NRC resource](https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm)
- Blei, D. M., Ng, A. Y. & Jordan, M. I. (2003). Latent Dirichlet Allocation. JMLR, 3, 993–1022. [Paper](https://www.jmlr.org/papers/v3/blei03a.html)
- [scikit-learn LDA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html)
- [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- Mimno et al. (2011). Optimizing Semantic Coherence in Topic Models. [Paper](https://aclanthology.org/D11-1024/)
